# V-ChartText: Demo Web Gradio & Triển khai Kiểm thử (Bước F: Tác vụ 33 - 34)
**Đề tài nghiên cứu**: Hệ thống trích xuất, kiểm chứng và sinh mô tả 3 cấp độ tích hợp đọc giọng nói cho biểu đồ giáo dục
**Tác giả**: Nhóm nghiên cứu AI & VLM
**Nội dung thực hiện**:
- **Tác vụ 33**: Lập trình Demo Web bằng Gradio với luồng tương tác hoàn chỉnh: Tải ảnh biểu đồ -> Trích xuất bảng số liệu & cờ cảnh báo [chưa chắc chắn] -> Sinh mô tả 3 cấp độ (Tổng quan 5s, Tóm tắt GD 30s, Phân tích chuyên sâu) -> Nghe đọc tiếng Việt (TTS).
- **Tác vụ 34**: Triển khai Demo (Hỗ trợ chạy Local, Google Colab với share=True, Hugging Face Spaces) & Chạy Smoke-test tự động kiểm thử toàn bộ luồng.

In [ ]:
# Cài đặt các thư viện cần thiết nếu chạy trên Google Colab
!pip install -q gradio gTTS pandas openpyxl pillow

In [ ]:
import os
import re
import json
import pandas as pd
from PIL import Image
from gtts import gTTS
import gradio as gr

print(f'Gradio version: {gr.__version__}')

## 1. Tích hợp Pipeline 4 bước (D21 - D28)
Hệ thống sử dụng pipeline 4 bước đã được kiểm chứng ở Bước D & E:
1. Trích xuất cấu trúc & bảng dữ liệu số liệu chuẩn hóa.
2. Kiểm tra tính hợp lệ & gắn cờ [chưa chắc chắn].
3. Sinh mô tả 3 cấp độ kiểm chứng (Grounded ViChartFact).
4. Chuẩn hóa số đọc và tổng hợp giọng nói tiếng Việt (TTS).

In [ ]:
# Đường dẫn thư mục dữ liệu
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(BASE_DIR, '..')) if os.path.basename(BASE_DIR) == 'notebooks' else BASE_DIR

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
DOCS_DIR = os.path.join(PROJECT_ROOT, 'docs')

BENCHMARK_PATH = os.path.join(DOCS_DIR, 'benchmark.xlsx')
if not os.path.exists(BENCHMARK_PATH):
    BENCHMARK_PATH = os.path.join(DOCS_DIR, 'Benchmark.xlsx')
if not os.path.exists(BENCHMARK_PATH):
    BENCHMARK_PATH = os.path.join(PROJECT_ROOT, 'benchmark.xlsx')

DATASET_DIR = os.path.join(DATA_DIR, 'dataset')
if not os.path.exists(DATASET_DIR):
    DATASET_DIR = os.path.join(PROJECT_ROOT, 'dataset')

print(f'Project root: {PROJECT_ROOT}')
print(f'Benchmark path: {BENCHMARK_PATH}')
print(f'Dataset dir: {DATASET_DIR}')


## 2. Kiểm thử Smoke-Test Tự động (Tác vụ 34)
Kiểm thử 5 mẫu biểu đồ đại diện cho 5 loại phổ biến: Cột đơn, Cột nhóm, Miền, Tròn, Đường.

In [ ]:
LOG_PATH = os.path.join(DATA_DIR, 'VChartText_Outputs', 'smoke_test_demo_log.csv')
if not os.path.exists(LOG_PATH):
    LOG_PATH = os.path.join(PROJECT_ROOT, 'VChartText_Outputs', 'smoke_test_demo_log.csv')

if os.path.exists(LOG_PATH):
    df_smoke = pd.read_csv(LOG_PATH)
    print('Kết quả Smoke-test đã thực hiện:')
    display(df_smoke)
else:
    print('Smoke test log chưa tồn tại, có thể chạy lại từ app.py hoặc smoke_test script.')


## 3. Khởi chạy Giao diện Web Demo Gradio (Tác vụ 33 & 34)
Chạy lệnh bên dưới để mở giao diện web. Tham số share=True sẽ tạo một đường link công khai (Gradio Live Link) có thời hạn 72 giờ, rất thuận tiện khi chạy trên Google Colab.

In [ ]:
# Đọc và chạy trực tiếp app.py
import sys
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from app import demo

# Khởi chạy demo với share=True để truy cập từ xa nếu chạy trên Colab
demo.launch(share=False, inbrowser=True)
